# OpenPlaque — memory-safe staged left-main and bifurcation tracking

This is the RAM-safe continuation of the left-main experiment. It reuses the already cached source evidence and RCA calibration, but replaces the full-volume coordinate grids and graph allocations with sparse candidate screening and local graph-search boxes.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_SOURCE_EVIDENCE = True
REUSE_RCA_CALIBRATION = True
REUSE_LEFT_OSTIUM = True
REUSE_LEFT_MAIN = True
REUSE_BIFURCATION_BRANCHES = True
REUSE_QC_FIGURES = True
REUSE_REPORT_PACKAGE = True


## Step 3 — Install this branch


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch left-main-bifurcation-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas psutil
import sys, gc, os, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label='RAM'):
    p=psutil.Process(os.getpid())
    print(f'{label}: process RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Activate the memory-safe graph implementation and create workflow


In [ ]:
from openplaque.left_main_memory_safe import patch_base_module, ALGORITHM_VERSION
patch_base_module()
from openplaque.left_main_bifurcation_workflow import LeftMainBifurcationWorkflow
reuse = {
    'source_evidence': REUSE_SOURCE_EVIDENCE,
    'rca_calibration': REUSE_RCA_CALIBRATION,
    'left_ostium': REUSE_LEFT_OSTIUM,
    'left_main': REUSE_LEFT_MAIN,
    'bifurcation_branches': REUSE_BIFURCATION_BRANCHES,
    'qc_figures': REUSE_QC_FIGURES,
    'report_package': REUSE_REPORT_PACKAGE,
}
wf = LeftMainBifurcationWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
print('Algorithm:', ALGORITHM_VERSION)
display(wf.cache_status())


## Step 5 — Reuse source evidence and RCA calibration


In [ ]:
wf.prepare_evidence()
wf.calibrate_rca()
ram('After evidence + RCA calibration')
display(wf.rca_cal)


## Step 6 — Memory-safe left-coronary ostium candidates
No full-volume `np.indices` array is created. Only a sparse aortic-wall shortlist is ray-tested.


In [ ]:
ostia = wf.detect_left_ostium()
gc.collect(); ram('After ostium search')
display(ostia.head(12))


## Step 7 — Memory-safe short left-main graph search + serial lumen QC
Graph search is restricted to a local physical box around each ostium and its candidate endpoints.


In [ ]:
left_main = wf.build_left_main()
gc.collect(); ram('After left-main search')
print('Left-main summary:')
display(wf.left_main_summary)
display(wf.left_main_qc_df)


## Step 8 — Bifurcation search from the accepted trunk endpoint
The distal search also uses sparse endpoint discovery and a local MCP bounding box. Failure to find an acceptable pair is reported but does not prevent report packaging.


In [ ]:
paths = wf.build_bifurcation_branches()
gc.collect(); ram('After branch search')
display(wf.summary_table())
print('Pair metadata:')
display(wf.branch_pair)


## Step 9 — QC figures


In [ ]:
figures = wf.plot_qc()
for f in figures: print('Saved:', f)
gc.collect(); ram('After QC figures')


## Step 10 — Package report


In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_MAIN_BIFURCATION_REPORT_BACK.zip')


After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**.
